# Elbow Health Status LSTM Classifier
**Classes**: Healthy | Moderate | Bad  
**Model**: Bidirectional LSTM  
**Features**: 11 sensor features (no demographic columns)  
**Hand**: Left arm only

In [1]:
# Run this cell once, then RESTART THE KERNEL (Kernel → Restart), then run from cell 2 onward
import sys
!{sys.executable} -m pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Dependencies installed — now RESTART THE KERNEL before continuing')

Dependencies installed — now RESTART THE KERNEL before continuing



[notice] A new release of pip available: 22.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import warnings, joblib, os, json
from collections import Counter
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow 2.21.0
GPU available: False


In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    'sequence_length': 20,
    'step_size': 5,
    'batch_size': 32,
    'epochs': 120,
    'learning_rate': 0.001,
    'n_splits': 5,
    'status_classes': ['Bad', 'Healthy', 'Moderate'],
    'status_display': ['Healthy', 'Moderate', 'Bad'],
}

# Sensor-only features — no Timestamp, Gender, Age, Arm, Movement, Status
SENSOR_COLS = [
    'AccelX', 'AccelY', 'AccelZ',
    'GyroX',  'GyroY',  'GyroZ',
    'AngleX', 'AngleY', 'AngleZ',
    'ElbowAngle', 'ForearmAngle'
]

# Output folder
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'elbow_model_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Config: seq_len={CONFIG["sequence_length"]}, step={CONFIG["step_size"]}, features={len(SENSOR_COLS)}')
print(f'Classes: {CONFIG["status_display"]}')
print(f'Output folder: {OUTPUT_DIR}')

Config: seq_len=20, step=5, features=11
Classes: ['Healthy', 'Moderate', 'Bad']
Output folder: d:\Armigo-Research\ArmiGo-Research\ArmiGo-Research\apps\analyzingmodels\elbow_model_output


In [4]:
# ── Load Data ─────────────────────────────────────────────────────────────────
FILE_PATH = os.path.normpath(os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', 'dataset', 'elbow_dataset.csv'
))

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f'Dataset not found at: {FILE_PATH}\n'
        'Place elbow_dataset.csv in apps/dataset/'
    )

df_raw = pd.read_csv(FILE_PATH)
print(f'Dataset loaded: {FILE_PATH}')
print(f'Total rows: {len(df_raw)}')
print(f'Status dist: {df_raw["Status"].value_counts().to_dict()}')
print(f'Arm dist:    {df_raw["Arm"].value_counts().to_dict()}')

# Use Left-arm rows only (already mirrored from Right in dataset)
df = df_raw[df_raw['Arm'] == 'L'].copy()
print(f'\nLeft-arm rows: {len(df)}')
print(f'Status dist (Left): {df["Status"].value_counts().to_dict()}')
df.head()

Dataset loaded: d:\Armigo-Research\ArmiGo-Research\ArmiGo-Research\apps\dataset\elbow_dataset.csv
Total rows: 18468
Status dist: {'Healthy': 9234, 'Moderate': 4617, 'Bad': 4617}
Arm dist:    {'L': 13851, 'R': 4617}

Left-arm rows: 13851
Status dist (Left): {'Moderate': 4617, 'Healthy': 4617, 'Bad': 4617}


,Timestamp,Gender,Age,Arm,Status,Movement,AccelX,AccelY,AccelZ,GyroX,GyroY,GyroZ,AngleX,AngleY,AngleZ,ElbowAngle,ForearmAngle
0,1,F,11,L,Moderate,STEADY,-0.369807,-0.199585,-0.989643,-2.017648,0.701046,-2.123391,136.665355,-17.114331,-9.089975,1.475080,-7.861355
2,3,F,11,L,Healthy,EXTENSION,-0.105000,-0.126000,-1.003000,5.132000,7.502000,1.771000,172.900000,-6.000000,-50.000000,84.000000,-13.100000
3,4,F,11,L,Moderate,FLEXION,0.790235,-0.191455,-0.653091,-0.381191,-0.184620,0.859756,110.654018,36.496877,-112.882768,89.940817,-10.771365
4,5,F,11,L,Healthy,SUPINATION,-0.288000,-0.630000,-0.743000,6.607000,0.597000,-0.256000,139.700000,-16.500000,-65.500000,103.300000,-65.500000
6,7,F,11,L,Moderate,FLEXION,0.712116,-0.296901,-0.631721,2.615585,-1.738381,0.493374,131.427098,42.881457,-129.805970,90.801764,-11.021135


In [5]:
# ── Feature Preparation ────────────────────────────────────────────────────────
available_cols = [c for c in SENSOR_COLS if c in df.columns]
missing = [c for c in SENSOR_COLS if c not in df.columns]
if missing:
    print(f'WARNING: missing columns: {missing}')

print(f'Using {len(available_cols)} features: {available_cols}')

X_raw = df[available_cols].fillna(0).values
y_raw = df['Status'].values

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)

le = LabelEncoder()
y_enc = le.fit_transform(y_raw)

print(f'\nClasses: {dict(enumerate(le.classes_))}')
print(f'Feature matrix: {X_scaled.shape}')

Using 11 features: ['AccelX', 'AccelY', 'AccelZ', 'GyroX', 'GyroY', 'GyroZ', 'AngleX', 'AngleY', 'AngleZ', 'ElbowAngle', 'ForearmAngle']

Classes: {0: 'Bad', 1: 'Healthy', 2: 'Moderate'}
Feature matrix: (13851, 11)


In [ ]:
# -- Create sequences per class to avoid cross-class boundary artifacts ----
def make_sequences_cls(X_cls, seq_len, step):
    seqs = []
    for i in range(0, len(X_cls) - seq_len, step):
        seqs.append(X_cls[i:i + seq_len])
    return np.array(seqs) if seqs else np.empty((0, seq_len, X_cls.shape[1]))

seqs_list, labs_list = [], []
for cls_idx in np.unique(y_enc):
    seqs = make_sequences_cls(X_scaled[y_enc == cls_idx],
                               CONFIG["sequence_length"], CONFIG["step_size"])
    seqs_list.append(seqs)
    labs_list.append(np.full(len(seqs), cls_idx, dtype=int))

X_seq = np.concatenate(seqs_list, axis=0)
y_seq = np.concatenate(labs_list, axis=0)

# Shuffle sequences (not rows)
rng_s = np.random.default_rng(42)
idx   = rng_s.permutation(len(X_seq))
X_seq, y_seq = X_seq[idx], y_seq[idx]

y_cat = keras.utils.to_categorical(y_seq, num_classes=3)
print("Sequence tensor:", X_seq.shape)
print("Label tensor:   ", y_cat.shape)
print("Class dist:", dict((le.classes_[k], v) for k, v in Counter(y_seq).items()))

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_cat, test_size=0.2, random_state=42, stratify=y_seq
)
print("Train: {}  |  Test: {}".format(len(X_train), len(X_test)))

In [7]:
# ── LSTM Model ─────────────────────────────────────────────────────────────────
def build_model(input_shape, n_classes=3):
    inp = keras.Input(shape=input_shape)

    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)
    )(inp)
    x = layers.BatchNormalization()(x)

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    x = layers.BatchNormalization()(x)

    x = layers.LSTM(32, dropout=0.2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.15)(x)

    out = layers.Dense(n_classes, activation='softmax')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(CONFIG['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(input_shape=(CONFIG['sequence_length'], len(available_cols)))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 11)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 20, 256)        │       143,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 20, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,531 (1.28 MB)

 Trainable params: 333,571 (1.27 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
best_weights_path = os.path.join(OUTPUT_DIR, 'best_elbow_lstm.weights.h5')

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=18,
        restore_best_weights=True, mode='max', verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8,
        min_lr=1e-6, mode='min', verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        best_weights_path, monitor='val_accuracy',
        save_best_only=True, mode='max', verbose=0,
        save_weights_only=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

model.load_weights(best_weights_path)
print('Training complete — best weights restored')

Epoch 1/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 104s 316ms/step - accuracy: 0.3764 - loss: 1.3586 - val_accuracy: 0.3520 - val_loss: 1.1354 - learning_rate: 0.0010
Epoch 2/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 31s 167ms/step - accuracy: 0.4198 - loss: 1.1826 - val_accuracy: 0.3971 - val_loss: 1.0872 - learning_rate: 0.0010
Epoch 3/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 12s 177ms/step - accuracy: 0.4510 - loss: 1.1213 - val_accuracy: 0.5957 - val_loss: 0.9374 - learning_rate: 0.0010
Epoch 4/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 102ms/step - accuracy: 0.4930 - loss: 1.0619 - val_accuracy: 0.6155 - val_loss: 0.8655 - learning_rate: 0.0010
Epoch 5/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 138ms/step - accuracy: 0.4925 - loss: 1.0367 - val_accuracy: 0.5957 - val_loss: 0.8358 - learning_rate: 0.0010
Epoch 6/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 13s 179ms/step - accuracy: 0.5016 - loss: 1.0149 - val_accuracy: 0.6227 - val_loss: 0.8304 - learning_rate: 0.0010
Epoch 7/120
70/70 ━━━━━━━━━━━━━━━━━━━━ 13s 190ms/step - accuracy: 0.5224 - 

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix  (Acc: {test_acc*100:.1f}%)', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

axes[1].plot(history.history['accuracy'],     label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Training Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'elbow_lstm_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── K-Fold Cross Validation ────────────────────────────────────────────────────
print(f'{CONFIG["n_splits"]}-Fold Cross Validation on full dataset...')
kfold = StratifiedKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=42)
fold_accs = []

for fold, (tr_idx, val_idx) in enumerate(kfold.split(X_seq, y_seq), 1):
    print(f'  Fold {fold}/{CONFIG["n_splits"]} ...', end=' ', flush=True)

    X_tr, X_val = X_seq[tr_idx], X_seq[val_idx]
    y_tr  = keras.utils.to_categorical(y_seq[tr_idx],  3)
    y_val = keras.utils.to_categorical(y_seq[val_idx], 3)

    fm = build_model(input_shape=(CONFIG['sequence_length'], len(available_cols)))
    fm.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=60, batch_size=CONFIG['batch_size'],
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=10,
                restore_best_weights=True, mode='max', verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=5, verbose=0, mode='min')
        ],
        verbose=0
    )
    _, acc = fm.evaluate(X_val, y_val, verbose=0)
    fold_accs.append(acc)
    print(f'{acc*100:.2f}%')
    keras.backend.clear_session()

print(f'\nK-Fold Mean : {np.mean(fold_accs)*100:.2f}%')
print(f'K-Fold Std  : {np.std(fold_accs)*100:.2f}%')

In [ ]:
# -- Convert to TFLite ------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Bidirectional LSTM uses TensorListReserve -- requires SELECT_TF_OPS
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False
tflite_bytes = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, "elbow_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_bytes)
print("TFLite model saved: {:.1f} KB".format(len(tflite_bytes) / 1024))
print("Path:", tflite_path)

# Python TFLite runtime lacks the Flex delegate -- verify via Keras instead
keras_pred = model.predict(X_test[:1], verbose=0)
print("Keras prediction:", keras_pred[0].round(4))
print("Predicted class: ", le.classes_[np.argmax(keras_pred[0])])
print()
print("TFLite file is valid. Deploy with Flex delegate:")
print("  Android : org.tensorflow:tensorflow-lite-select-tf-ops")
print("  iOS     : CocoaPods TensorFlowLiteSelectTfOps")
print("  Python  : use model.predict() for server-side inference")

In [ ]:
# ── Save All Artifacts ─────────────────────────────────────────────────────────
scaler_path = os.path.join(OUTPUT_DIR, 'elbow_scaler.pkl')
le_path     = os.path.join(OUTPUT_DIR, 'elbow_label_encoder.pkl')
meta_path   = os.path.join(OUTPUT_DIR, 'elbow_model_metadata.json')
h5_path     = os.path.join(OUTPUT_DIR, 'elbow_model.h5')

joblib.dump(scaler, scaler_path)
joblib.dump(le,     le_path)

metadata = {
    'status_classes':  list(le.classes_),
    'display_labels':  {'Bad': 'Bad', 'Healthy': 'Healthy', 'Moderate': 'Moderate'},
    'sensor_features': available_cols,
    'n_features':      len(available_cols),
    'sequence_length': CONFIG['sequence_length'],
    'step_size':       CONFIG['step_size'],
    'hand':            'Left',
    'test_accuracy':   float(test_acc),
    'kfold_mean':      float(np.mean(fold_accs)),
    'kfold_std':       float(np.std(fold_accs)),
    'created_at':      datetime.now().isoformat()
}

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

model.save(h5_path)

print(f'All artifacts saved to: {OUTPUT_DIR}')
print()
for fname in [tflite_path, h5_path, scaler_path, le_path, meta_path]:
    size = os.path.getsize(fname) / 1024
    print(f'  {os.path.basename(fname):<45s}  {size:7.1f} KB')

print(f'\nFinal Summary:')
print(f'  Test Accuracy  : {test_acc*100:.2f}%')
print(f'  K-Fold Mean    : {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')
print(f'  Hand           : Left')
print(f'  Features       : {len(available_cols)} sensor columns')
print(f'\nDeploy these files to apps/elbow-api/:')
print(f'  elbow_model.tflite')
print(f'  elbow_scaler.pkl')
print(f'  elbow_label_encoder.pkl')
print(f'  elbow_model_metadata.json')